In [ ]:
# 🙊 Toxic Language Detector — Prototype
Using **`detoxify`** with the `multilingual` model (`unitary/multilingual-toxic-xlm-roberta`) to score text across 6 toxicity dimensions and trigger notifications based on a configurable threshold.

In [ ]:
# ── 1. Imports ────────────────────────────────────────────────────────────────
import pandas as pd
from detoxify import Detoxify

## 2. Load the Multilingual Model
`Detoxify('multilingual')` downloads and caches `unitary/multilingual-toxic-xlm-roberta` from HuggingFace on first run.

In [ ]:
# Downloads ~1 GB on first run, then cached locally
model = Detoxify('multilingual')
print("✅ Model loaded.")

## 3. Sample Texts
A diverse set of inputs — clean, borderline, and clearly toxic — to exercise all score dimensions.

In [ ]:
sample_texts = [
    # --- Clean ---
    "I really enjoyed the documentary last night, very informative.",
    "Can you help me understand how neural networks work?",
    "The weather today is absolutely beautiful!",
    # --- Borderline ---
    "I disagree with your opinion, I think you're completely wrong.",
    "This code is garbage and whoever wrote it should be ashamed.",
    "Stop being so stupid and just follow the instructions.",
    # --- Toxic ---
    "I hate you, you're worthless and should disappear.",
    "People like you are a disease to society.",
    "Go kill yourself, nobody wants you here.",
]

## 4. Score & Notification System
Scores are in `[0, 1]`. We define three alert levels based on the highest score across all toxicity dimensions:

| Toxicity Score | Alert Level |
|---|---|
| `< 0.30` | ✅ Clean |
| `0.30 – 0.70` | ⚠️ Warning |
| `> 0.70` | 🚨 High Alert |

In [ ]:
WARNING_THRESHOLD  = 0.30   # above this → ⚠️  Warning
ALERT_THRESHOLD    = 0.70   # above this → 🚨  High Alert

def get_alert_level(max_score: float) -> str:
    if max_score >= ALERT_THRESHOLD:
        return "🚨  HIGH ALERT"
    elif max_score >= WARNING_THRESHOLD:
        return "⚠️   WARNING"
    else:
        return "✅  CLEAN"

def trigger_notification(text: str, scores: dict) -> dict:
    """Return a notification dict with alert level and the top offending score."""
    max_label = max(scores, key=scores.get)
    max_score = scores[max_label]
    level     = get_alert_level(max_score)
    return {
        "text":      text[:80] + ("…" if len(text) > 80 else ""),
        "level":     level,
        "top_label": max_label,
        "top_score": round(max_score, 4),
    }

# ── Run inference on all samples ──────────────────────────────────────────────
raw_scores = model.predict(sample_texts)          # dict of label → list of floats

# Reshape into per-text dicts
per_text_scores = [
    {label: raw_scores[label][i] for label in raw_scores}
    for i in range(len(sample_texts))
]

notifications = [trigger_notification(t, s) for t, s in zip(sample_texts, per_text_scores)]

# Pretty-print notifications
print(f"{'ALERT':<20} {'TOP LABEL':<22} {'SCORE':>6}   TEXT")
print("─" * 100)
for n in notifications:
    print(f"{n['level']:<20} {n['top_label']:<22} {n['top_score']:>6.4f}   {n['text']}")

## 5. Full Score Breakdown
All six toxicity dimensions per sample as a styled DataFrame.

In [ ]:
df = pd.DataFrame(raw_scores, index=sample_texts).round(4)
df.index.name = "text"

# Add alert column
df.insert(0, "alert", [n["level"] for n in notifications])

df.style.background_gradient(
    subset=list(raw_scores.keys()),
    cmap="RdYlGn_r",
    vmin=0, vmax=1
).format({col: "{:.4f}" for col in raw_scores.keys()})

## 6. Try Your Own Text
Edit `my_text` below and re-run the cell to see the toxicity score and notification in real time.

In [ ]:
# ✏️  Change this text to anything you want to test
my_text = "You are such an idiot, I can't believe you exist."

scores  = model.predict(my_text)
notif   = trigger_notification(my_text, scores)

print(f"\n📝 Input : {my_text}")
print(f"🔔 Alert : {notif['level']}")
print(f"📊 Top   : {notif['top_label']} = {notif['top_score']:.4f}\n")
print("Full scores:")
pd.Series(scores).round(4).sort_values(ascending=False).to_frame("score")